In [20]:
# Do Not Modified. Generated by iantirta.com
import os
os.environ['IANTIRTA_API_KEY'] = '014d799e2f289b90c85b534f0b2d61418685e5013e3487cec83f4b2ea36cc491'

iantirta_url = "http://karaoke.iantirta.com"
worker_name = "Kaggle Worker `crows.nony@gmail.com`"
worker_provider = "kaggle"

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# Test
import requests
res = requests.post(f"{iantirta_url}/karaoke/task", json={
    "params": {},
}, allow_redirects=False)
import rich
if res.is_redirect:
    new_url = res.headers["Location"]
    res = requests.post(new_url, json={
        "params": {},
    })
    print(res.json())

{'jsonrpc': '2.0', 'id': None, 'error': {'code': 0, 'message': 'Sigil Server Error', 'data': {'name': 'builtins.TypeError', 'message': "KaraokeController.karaoke_task() missing 4 required positional arguments: 'worker_name', 'worker_provider', 'access_token', and 'status_to_fetch'", 'arguments': ["KaraokeController.karaoke_task() missing 4 required positional arguments: 'worker_name', 'worker_provider', 'access_token', and 'status_to_fetch'"], 'context': {}, 'debug': 'Traceback (most recent call last):\n  File "/usr/lib/python3/dist-packages/sigil/http.py", line 2328, in _serve_db\n    return service_model.retrying(serve_func, env=self.env)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/lib/python3/dist-packages/sigil/service/model.py", line 187, in retrying\n    result = func()\n             ^^^^^^\n  File "/usr/lib/python3/dist-packages/sigil/http.py", line 2383, in _serve_ir_http\n    response = self.dispatcher.dispatch(rule.endpoint, args)\n             

In [23]:
#@title Karaoke Plus
from __future__ import annotations

import os
from pathlib import Path
import subprocess
import typing as t
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
import concurrent.futures
import requests
import argparse
import logging

import kplus
from kplus import env, config
from kplus.pipelines.separate import BaseSeparator, DemucsSeparator
from kplus.pipelines.utils import ASRResult
from kplus.pipelines import (
    ensure_file,
    transcribe,
    align,
    align2ref,
    detect_audio_activity,
)
from kplus.pipelines.asr.hf import HFModel
from kplus.tools.render import Render


# Logging Setup
opt = argparse.Namespace(log_level="debug")
config.parse_config(opt, setup_logging=True)
logger = logging.getLogger(__name__)

# Secret getter
def get_secret(key: str, *args, **kwargs):
    """ Return a secret defaulting to os.environ.
    """
    try:
        if kplus.env.is_kaggle:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key, *args, *kwargs)
        elif kplus.env.is_colab:
            from google.colab import userdata
            return userdata.get(key, *args, *kwargs)
    except:
        pass
    return os.environ.get(key, *args, **kwargs)


TaskStatus: t.TypeAlias = t.Literal["waiting", "processing", "completed", "error"]
TaskType: t.TypeAlias = t.Literal["basic", "plus"]

if t.TYPE_CHECKING:
    from kplus.pipelines.utils import ASRResult, AudioSegment
    from kplus.pipelines.asr.hf.mixin import ASRMixin


@dataclass(slots=True)
class Task:
    """ Hold Task Type """
    id: int
    title: str
    artist: str
    duration: float
    lyrics: str
    status: TaskStatus
    url: str
    karaoke_type: TaskType

    # Processing helper
    audiopath: str | None = field(init=False, default=...)
    instpath: str | None = field(init=False, default=...)
    samplerate: int | None = field(init=False, default=...)
    audiosegments: AudioSegment | None = field(init=False, default=...)
    result: ASRResult | None = field(init=False, default=...)

    error: str | None = field(init=False, default=...)
    videopath: str | None = field(init=False, default=...)
    karaokepath: str | None = field(init=False, default=...)

    @classmethod
    def from_dict(cls, data: dict) -> Task:
        return cls(
            id=data["id"],
            title=data["title"],
            artist=data["artist"],
            duration=data["duration"],
            lyrics=data["lyrics"],
            status=data["status"],
            url=data["url"],
            karaoke_type=data["karaoke_type"],
        )

    def update_status(self, status: str, message: str):
        """ This should be update error
        """
        self.status = status
        self.error = message


tasks: list[Task] = []


class APIError(Exception):
    """ Handle API Error """


class IantirtaAPI:
    """ API Needed for connection to iantirta.com. """
    def __init__(self):
        self.api_token = get_secret("IANTIRTA_API_KEY")

    def post(self, url: str, params: dict):
        data = {
            "params": {
                "worker_name": worker_name,
                "worker_provider": worker_provider,
                "access_token": self.api_token,
            }
        }
        if "params" in params:
            data["params"].update(params["params"])
        else:
            data["params"].update(params)
        headers = {
            "Authorization": f"bearer {self.api_token}",
        }
        try:
            res = requests.post(url, headers=headers, json=data, allow_redirects=False)
            res.raise_for_status()
            if res.is_redirect:
                new_url = res.headers["Location"]
                res = requests.post(new_url, headers=headers, json=data)
                res.raise_for_status()
            
            res = res.json()
            if res.get("error", False):
                raise APIError("API Error:", res)
            else:
                return res["result"]
        except Exception:
            raise

    def get_task(self, status_to_fetch: TaskStatus) -> list[Task]:
        api_url = f"{iantirta_url}/karaoke/task"
        params = {"params": {
            "status_to_fetch": status_to_fetch,
        },}
        res = self.post(api_url, params)

        global tasks
        tasks = [Task.from_dict(r) for r in res]
        return tasks

class DriveAPI:
    """ API Needed for connection to google.drive. """
    SCOPES = ['https://www.googleapis.com/auth/drive']  # noqa: RUF012
    

class CloudflareAPI:
    """ API Needed for connection to cloudflare storage. """

class KaraokeWorker:
    def __init__(self):
        self.cookiefile = "cookies.txt"
        self.iantirta_api = IantirtaAPI()
        self.iantirta_api.get_task("waiting")
        

    def _run_render(
        self,
        task: Task,
        instrument_path: str,
        result: ASRResult | None = None
    ) -> None:
        task.karaokepath = Render(with_ass=task.karaoke_type == 'plus').render(
            video_filepath=task.videopath,
            inst_path=instrument_path,
            duration=task.duration,
            result=result,
            output_path=None,
        )
        task.status = "completed"

    def _run(
        self,
        task: Task,
        separator: DemucsSeparator,
        executor: ThreadPoolExecutor,
    ) -> None:
        # Normalize to wav
        inputpath = Path(task.videopath).expanduser().resolve()
        mixpath = inputpath.stem + ".wav"
        subprocess.run([
            "ffmpeg", "-y", "-i",
            str(inputpath), "-vn",
            "-ar", str(separator.sr),
            "-ac", str(separator.ac),
            str(mixpath)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Run separation
        sep_result = separator.separate(mixpath)
        instrument_path = sep_result.inst_path

        if os.path.exists(mixpath):
            os.unlink(mixpath)
        
        if task.karaoke_type == "basic":
            # Imidiately pass it to render & upload
            executor.submit(
                self._run_render,
                task,
                instrument_path,
            )
        elif task.karaoke_type == "plus":
            # Pass here since it needs to be run separately.
            task.audiopath = sep_result.vocs_path
            task.instpath = instrument_path
            task.samplerate = sep_result.sr
        else:
            raise RuntimeError(
                f"Karaoke Type `{task.karaoke_type}` "
                "isn't supported yet."
            )
    
    def run(self) -> None:
        """ Pipeline
            * basic_tasks: (download, separate, render)
            * plust_tasks: (download, separate, transcribe, render)
        """
        render_executor = ThreadPoolExecutor(max_workers=3)
        separator = BaseSeparator.from_options(
            demucs="mdx_extra_q",
            overlap=0.75,
            segment=200,
            shifts=1,
            num_workers=0,
        )

        with ThreadPoolExecutor(num_workers=5) as executor:
            iter_task = iter(tasks)
            active_futures: dict[t.Any, Task] = {}
            for _ in range(5):
                if task := next(iter_task, None):
                    future = executor.submit(
                        ensure_file,
                        inputpath=task.url,
                        external_id=task.id,
                        no_lyrics=task.karaoke_type == "basic",
                        cookiefile=self.cookiefile,
                    )
                    active_futures[future] = task
            while active_futures:
                done, _ = concurrent.futures.wait(
                    active_futures.keys(),
                    return_when=concurrent.futures.FIRST_COMPLETED
                )
                for future in done:
                    task = active_futures.pop(future)
                    if next_task := next(iter_task, None):
                        new_future = executor.submit(
                            ensure_file,
                            inputpath=next_task.url,
                            external_id=next_task.id,
                            no_lyrics=next_task.karaoke_type == "basic",
                            cookiefile=self.cookiefile,
                        )
                        active_futures[new_future] = next_task
                    try:
                        info = future.result()
                        assert info.duration == task.duration, (
                            "Duration Missmatch between "
                            "server side and worker side, "
                            f"{info.duration} | {task.duration}"
                        )
                        assert info.title == task.title, (
                            "Title Missmatch between "
                            "server side and worker side, "
                            f"{info.title} | {task.title}"
                        )
                        assert info.artist == task.artist, (
                            "Artist Missmatch between "
                            "server side and worker side, "
                            f"{info.artist} | {task.artist}"
                        )
                        task.videopath = info.filepath
                        self._run(task, separator=separator, executor=render_executor)
                    except Exception as e:
                        logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                        task.update_status("failed", str(e))

        del separator.model, separator
        env.clean()

        # Audio Detection
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    audio_result = detect_audio_activity(
                        audio=task.audiopath,
                        sr=task.samplerate,
                        no_show=True,
                        resample=False,
                        use_html=False,
                    )
                    task.audiosegments = audio_result.segments
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        # ASR
        transcriber: ASRMixin = HFModel.from_pretrained(
            "Qwen/Qwen3-ASR-1.7B-hf",
            max_inference_batch_size=1,
            num_beams=4,
        )
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    asr_result = transcriber.transcribe(
                        audio=task.audiopath,
                        audiosegments=task.audiosegments,
                        contexts=task.lyrics,
                        languages=None,
                        return_timestamps=True,
                    )
                    task.result = asr_result
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        del transcriber.model, transcriber
        env.clean()

        # into Lyrics
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    ref_result, new_audiosegments = align2ref(
                        task.result,
                        task.lyrics,
                        task.audiosegments,
                        raise_if_not_reliable=False
                    )
                    task.result = ref_result
                    task.audiosegments = new_audiosegments
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))

        # Alignment
        aligner: ASRMixin = HFModel.from_pretrained("facebook/mms-1b-all",)
        for task in tasks:
            if task.karaoke_type == "plus":
                try:
                    align_results = aligner.align(
                        audio=task.audiopath,
                        audiosegments=task.audiosegments,
                        hypothesis=task.result.texts,
                        languages=None,
                    )
                    align_results.to_line_idx(task.lyrics)
                    task.result = align_results.populate_ass()
                    render_executor.submit(
                        self._run_render,
                        task,
                        task.instpath,
                        task.result,
                    )
                except Exception as e:
                    logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                    task.update_status("failed", str(e))
        del aligner.model, aligner.processor, aligner
        env.clean()

        render_executor.shutdown()
        
        logger.info("Complete: All tasks processed through pipeline.")
        logger.info("complete")


KaraokeWorker().run()

╭─────────────────────────────────────────────────── Karaoke+ ────────────────────────────────────────────────────╮
│                                                                                                                 │
│   _  __                     _                                                                                   │
│  | |/ /__ _ _ __ __ _  ___ | | _____   _                                                                        │
│  | ' // _` | '__/ _` |/ _ \| |/ / _ \_| |_                                                                      │
│  | . \ (_| | | | (_| | (_) |   <  __/_   _|                                                                     │
│  |_|\_\__,_|_|  \__,_|\___/|_|\_\___| |_|                                                                       │
│                                                                                                                 │
│ Author     : iantirta.com                                                                                       │
│ Version    : 4.1rc                                                                                              │
│ Session    : DESKTOP-9QJ9V6V-75D6F0BA2FC3                                                                       │
│ Platform   : Windows-10-10.0.19045-SP0                                                                          │
│ System     : windows                                                                                            │
│ OS         : nt                                                                                                 │
│ Environment: local                                                                                              │
│ Python     : 3.12.10                                                                                            │
│                                                                                                                 │
╰───────────────────────────────────────────────────── Ready ─────────────────────────────────────────────────────╯

[09/21/26 14:05:58] [PID: 12616] ─                                               ]8;id=1869967;file://C:\Users\USER\Downloads\JAZPIPER_AUTOMATION\KaraokePlus\kplus\pipelines\separate\demucs.py\kplus.pipelines.separate.demucs]8;;\]8;id=1869968;file://C:\Users\USER\Downloads\JAZPIPER_AUTOMATION\KaraokePlus\kplus\pipelines\separate\demucs.py#28\:28]8;;\
 ↳ INFO     Chosen Separator Class: (Demucs)

[09/21/26 14:06:03] [PID: 12616] ─                                                                  ]8;id=1869973;file://c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\warnings.py\py.warnings]8;;\]8;id=1869974;file://c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\warnings.py#112\:112]8;;\
 ↳ WARNING  c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\rich\live.py:260: UserWarning: 
            install "ipywidgets" for Jupyter support                                                               
              warnings.warn('install "ipywidgets" for Jupyter support')

TODO: Process is stopping by signal SIGINT <frame at 0x000002BE8D6298C0, file 
'c:\\Users\\USER\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\diffq\\diffq.py', line 281, code
_unquantize_param>

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactive │
│ shell.py:3701 in run_code                                                                        │
│                                                                                                  │
│   3698 │   │   │   │   if async_:                                                                │
│   3699 │   │   │   │   │   await eval(code_obj, self.user_global_ns, self.user_ns)               │
│   3700 │   │   │   │   else:                                                                     │
│ ❱ 3701 │   │   │   │   │   exec(code_obj, self.user_global_ns, self.user_ns)                     │
│   3702 │   │   │   finally:                                                                      │
│   3703 │   │   │   │   # Reset our crash handler in place                                        │
│   3704 │   │   │   │   sys.excepthook = old_excepthook                                           │
│                                                                                                  │
│ in <module>:384                                                                                  │
│                                                                                                  │
│   381 │   │   logger.info("complete")                                                            │
│   382                                                                                            │
│   383                                                                                            │
│ ❱ 384 KaraokeWorker().run()                                                                      │
│   385                                                                                            │
│                                                                                                  │
│ in run:234                                                                                       │
│                                                                                                  │
│   231 │   │   │   * plust_tasks: (download, separate, transcribe, render)                        │
│   232 │   │   """                                                                                │
│   233 │   │   render_executor = ThreadPoolExecutor(max_workers=3)                                │
│ ❱ 234 │   │   separator = BaseSeparator.from_options(                                            │
│   235 │   │   │   demucs="mdx_extra_q",                                                          │
│   236 │   │   │   overlap=0.75,                                                                  │
│   237 │   │   │   segment=200,                                                                   │
│                                                                                                  │
│ C:\Users\USER\Downloads\JAZPIPER_AUTOMATION\KaraokePlus\kplus\pipelines\separate\__init__.py:32  │
│ in from_options                                                                                  │
│                                                                                                  │
│   29 │   │   │   ), demucs_modelname, uvr_modelname)                                             │
│   30 │   │   modelclass = self.modelclass["demucs" if demucs_modelname else "uvr"]               │
│   31 │   │   modelname = demucs_modelname if demucs_modelname else uvr_modelname                 │
│ ❱ 32 │   │   return modelclass(modelname, **options)                                             │
│   33                                                                                             │
│   34                                                                                             │
│   35 def separate_song(info: DownloadResult, **options) -> 

[09/21/26 14:06:09] [PID: 12616] ─                                                                  ]8;id=1869977;file://c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\warnings.py\py.warnings]8;;\]8;id=1869978;file://c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\warnings.py#112\:112]8;;\
 ↳ WARNING  c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.…
            UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.                                                   
              warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)